In [2]:
from sklearn.decomposition import PCA
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score)
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

In [ ]:
"""
Multi-Omics Feature Selection & Classification Pipeline
========================================================
Input : 4 CSV/TSV files  (rows = features, columns = samples)
        mrna.csv | mirna.csv | methylation.csv | cnv.csv
        A labels file: labels.csv  (one column, one row per sample, same order)

Correct leakage-free architecture
-----------------------------------
For each of the 5 outer folds:
  ├── TRAIN FOLD only:
  │     Step 1 – Autoencoder    → score and keep informative features
  │     Step 2 – LR-RFE         → rank surviving features
  │     Step 3 – IFS with LR    → find optimal feature count  (inner CV on train)
  │     Step 4 – SBS with LR    → refine feature set          (inner CV on train)
  │     Step 5 – Fit final XGBoost on train with selected features
  └── TEST FOLD only:
        Evaluate: accuracy, F1-macro, precision-macro, recall-macro

Report mean ± std of all metrics across the 5 folds.

NOTE: Scaling (StandardScaler) is also fit on train fold only and applied
      to test fold — no leakage at any stage.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0.  CONFIGURATION  –  edit paths / hyper-parameters here
# ─────────────────────────────────────────────────────────────────────────────
DATA_FILES = {
    "mrna":        "Main_Dataset/Classification_datasets/GS-BRCA/Aligned/BRCA_mRNA_aligned.csv",
    "mirna":       "Main_Dataset/Classification_datasets/GS-BRCA/Aligned/BRCA_miRNA_aligned.csv",
    "methylation": "Main_Dataset/Classification_datasets/GS-BRCA/Aligned/BRCA_Methy_aligned.csv",
    "cnv":         "Main_Dataset/Classification_datasets/GS-BRCA/Aligned/BRCA_CNV_aligned.csv",
}
LABELS_FILE = "Main_Dataset/Classification_datasets/GS-BRCA/Aligned/BRCA_label_num.csv"   # single column, no header OR header "label"

SEP          = ","
RANDOM_STATE = 42
N_FOLDS      = 5           # outer CV folds
N_INNER      = 4             # inner CV folds used inside IFS / SBS

# L1 LR (Step 1)
L1_C        = 1           # smaller → sparser solution
L1_SOLVER   = "liblinear"
L1_MAX_ITER = 5000

# General LR (Steps 2–5)
LR_C        = 1.0
LR_SOLVER   = "lbfgs"
LR_MAX_ITER = 5000

RFE_STEP    = 1       

# General LR (Steps 2–5)
LR_C        = 1.0
LR_SOLVER   = "lbfgs"
LR_MAX_ITER = 5000

RFE_STEP    = 1              # features eliminated per RFE iteration

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def load_omics(path: str, sep: str = ",") -> pd.DataFrame:
    """Load feature x sample CSV; return sample x feature (transposed)."""
    df = pd.read_csv(path, sep=sep, index_col=0)
    return df.T


def load_labels(path: str, sep: str = ",") -> np.ndarray:
    df = pd.read_csv(path, sep=sep, header= 0)
    return df.iloc[:, 0].values


def make_lr(C=LR_C, penalty="l2", solver=LR_SOLVER, max_iter=LR_MAX_ITER):
    return LogisticRegression(
        C=C, penalty=penalty, solver=solver,
        max_iter=max_iter, random_state=RANDOM_STATE,
        multi_class="auto",
    )


def make_autoencoder(input_dim: int):
    """Build a shallow autoencoder using an MLP regressor."""
    from sklearn.neural_network import MLPRegressor

    bottleneck = max(2, min(64, int(input_dim * AE_HIDDEN_RATIO)))
    return MLPRegressor(
        hidden_layer_sizes=(bottleneck,),
        activation="relu",
        solver="adam",
        alpha=AE_ALPHA,
        learning_rate_init=AE_LR_INIT,
        max_iter=AE_MAX_ITER,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        tol=1e-4,
    )


def step1_autoencoder_selection(X_train_s: np.ndarray,
                                y_train:   np.ndarray,
                                feature_names: list) -> tuple:
    """
    Fit a shallow autoencoder on scaled train data and keep the most
    influential features based on encoder/decoder weight norms.
    """
    ae = make_autoencoder(X_train_s.shape[1])
    ae.fit(X_train_s, X_train_s)
    w_in  = ae.coefs_[0]
    w_out = ae.coefs_[1]
    importance = np.linalg.norm(w_in, axis=1) * np.linalg.norm(w_out, axis=0)
    keep_n = max(1, int(np.ceil(AE_KEEP_RATIO * len(feature_names))))
    keep_idx = np.argsort(-importance)[:keep_n]
    mask = np.zeros(len(feature_names), dtype=bool)
    mask[keep_idx] = True
    selected = [fn for fn, m in zip(feature_names, mask) if m]
    return mask, selected


def make_xgb(y_train: np.ndarray) -> XGBClassifier:
    """Create XGBoost classifier with binary/multiclass-safe settings."""
    n_classes = int(np.unique(y_train).shape[0])
    params = {
        "n_estimators": 400,
        "max_depth": 7,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "tree_method": "hist",
    }
    if n_classes > 2:
        params.update({"objective": "multi:softprob",
                       "num_class": n_classes,
                       "eval_metric": "mlogloss"})
    else:
        params.update({"objective": "binary:logistic",
                       "eval_metric": "logloss"})
    return XGBClassifier(**params)


def impute_mean(X_train: np.ndarray, X_test: np.ndarray) -> tuple:
    """Mean-impute NaNs using train statistics only."""
    col_means = np.nanmean(X_train, axis=0)
    col_means  = np.where(np.isnan(col_means), 0.0, col_means)
    for X in (X_train, X_test):
        nans = np.where(np.isnan(X))
        X[nans] = np.take(col_means, nans[1])
    return X_train, X_test


def scale(X_train: np.ndarray, X_test: np.ndarray) -> tuple:
    """Fit StandardScaler on train, apply to both splits."""
    sc = StandardScaler()
    return sc.fit_transform(X_train), sc.transform(X_test)


def inner_cv_accuracy(X: np.ndarray, y: np.ndarray) -> float:
    """Mean accuracy of LR using N_INNER-fold stratified CV."""
    n_splits = min(N_INNER, int(np.min(np.bincount(y.astype(int)))))
    n_splits = max(n_splits, 2)
    cv   = StratifiedKFold(n_splits=n_splits, shuffle=True,
                           random_state=RANDOM_STATE)
    accs = []
    for tr, va in cv.split(X, y):
        X_tr, X_va = X[tr], X[va]
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr)
        X_va = sc.transform(X_va)
        lr = make_lr()
        lr.fit(X_tr, y[tr])
        accs.append(accuracy_score(y[va], lr.predict(X_va)))
    return float(np.mean(accs))


# ─────────────────────────────────────────────────────────────────────────────
# Step 1 – Autoencoder feature selection  (train only)
# ─────────────────────────────────────────────────────────────────────────────

def step1_l1_selection(X_train_s: np.ndarray,
                       y_train:   np.ndarray,
                       feature_names: list) -> tuple:
    """
    Fit L1 LR on scaled train data.
    Returns boolean mask and list of surviving feature names.
    """
    lr_l1 = LogisticRegression(
        penalty="l1", C=L1_C, solver=L1_SOLVER,
        max_iter=L1_MAX_ITER, random_state=RANDOM_STATE,
        multi_class="auto",
    )
    lr_l1.fit(X_train_s, y_train)
    importance = np.abs(lr_l1.coef_).max(axis=0)
    mask       = importance > 0
    selected   = [fn for fn, m in zip(feature_names, mask) if m]
    return mask, selected


def step2_rfe_order(X_train_s: np.ndarray,
                    y_train:   np.ndarray) -> np.ndarray:
    """
    Run RFE down to 1 feature on scaled train data.
    Returns argsort of RFE ranking (best column index first).
    """
    rfe = RFE(estimator=make_lr(), n_features_to_select=1, step=RFE_STEP)
    rfe.fit(X_train_s, y_train)
    return np.argsort(rfe.ranking_)


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 – Incremental Feature Selection  (train only, inner CV)
# ─────────────────────────────────────────────────────────────────────────────

def step3_ifs(X_train_ranked: np.ndarray,
              y_train:        np.ndarray,
              ranked_features: list) -> tuple:
    """
    Add one feature at a time in RFE rank order.
    Pick k that maximises inner-CV accuracy on the training fold.
    Returns (best_k, optimal_feature_names).
    """
    best_score, best_k = -1.0, 1
    for k in range(1, X_train_ranked.shape[1] + 1):
        score = inner_cv_accuracy(X_train_ranked[:, :k], y_train)
        if score > best_score:
            best_score, best_k = score, k
    return best_k, ranked_features[:best_k]


# ─────────────────────────────────────────────────────────────────────────────
# Step 4 – Sequential Backward Selection  (train only, inner CV)
# ─────────────────────────────────────────────────────────────────────────────

def step4_sbs(X_train_opt: np.ndarray,
              y_train:     np.ndarray,
              optimal_features: list) -> tuple:
    """
    Greedily remove features when removal does not decrease inner-CV accuracy.
    Returns (list of kept column indices, final_feature_names).
    """
    current_cols     = list(range(X_train_opt.shape[1]))
    current_features = list(optimal_features)
    baseline         = inner_cv_accuracy(X_train_opt[:, current_cols], y_train)

    improved = True
    while improved and len(current_cols) > 1:
        improved   = False
        best_score = baseline
        best_drop  = -1

        for i in range(len(current_cols)):
            trial = current_cols[:i] + current_cols[i+1:]
            score = inner_cv_accuracy(X_train_opt[:, trial], y_train)
            if score >= best_score:
                best_score = score
                best_drop  = i

        if best_drop >= 0:
            current_cols.pop(best_drop)
            current_features.pop(best_drop)
            baseline = best_score
            improved = True

    return current_cols, current_features


# ─────────────────────────────────────────────────────────────────────────────
# MAIN  –  leakage-free outer loop
# ─────────────────────────────────────────────────────────────────────────────

def main():
    # ── Load & merge data ──────────────────────────────────────────────────
    print("Loading omics data ...")
    dfs = {}
    for name, path in DATA_FILES.items():
        if not Path(path).exists():
            print(f"  [WARN] '{path}' not found – skipping '{name}'")
            continue
        dfs[name] = load_omics(path, SEP)
        print(f"  {name:<15}: {dfs[name].shape[0]} samples, "
              f"{dfs[name].shape[1]} features")

    if not dfs:
        print("\n[ERROR] No data files found. Set correct paths in CONFIG.")
        sys.exit(1)

    merged = pd.concat(list(dfs.values()), axis=1, join="inner")
    merged.columns = [
        f"{omic}__{feat}"
        for omic, df in dfs.items()
        for feat in df.columns
    ]
    print(f"\nMerged matrix : {merged.shape[0]} samples x "
          f"{merged.shape[1]} features")

    if not Path(LABELS_FILE).exists():
        print(f"\n[ERROR] Labels file '{LABELS_FILE}' not found.")
        sys.exit(1)

    y             = load_labels(LABELS_FILE, SEP)
    feature_names = list(merged.columns)
    X             = merged.values.astype(float)

    if len(y) != X.shape[0]:
        print(f"[ERROR] Label count ({len(y)}) != sample count ({X.shape[0]}).")
        sys.exit(1)

    # ── Outer 5-fold CV ────────────────────────────────────────────────────
    outer_cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)

    fold_metrics  = []
    fold_features = []

    print("\n" + "="*65)
    print("OUTER 5-FOLD CROSS-VALIDATION")
    print("Train/test split is performed FIRST; all feature selection")
    print("steps (1-4) run exclusively on the training fold.")
    print("="*65)

    for fold_idx, (train_idx, test_idx) in enumerate(
            outer_cv.split(X, y), start=1):

        print(f"\n{'─'*65}")
        print(f"  FOLD {fold_idx}/{N_FOLDS}  "
              f"(train={len(train_idx)}, test={len(test_idx)})")
        print(f"{'─'*65}")

        # ── 0. Split ───────────────────────────────────────────────────────
        X_train_raw = X[train_idx].copy()
        X_test_raw  = X[test_idx].copy()
        y_train     = y[train_idx]
        y_test      = y[test_idx]

        # ── 0a. Impute NaNs (train statistics only) ────────────────────────
        X_train_raw, X_test_raw = impute_mean(X_train_raw, X_test_raw)

        # ── 0b. Scale (train statistics only) ─────────────────────────────
        X_train_s, X_test_s = scale(X_train_raw, X_test_raw)

        # ── STEP 1 – Autoencoder selection (train only) ───────────────────
        mask, sel_features = step1_l1_selection(
            X_train_s, y_train, feature_names)

        X_train_sel = X_train_s[:, mask]
        X_test_sel  = X_test_s[:, mask]
        print(f"  Step 1 | LR selection   : "
              f"{len(feature_names)} -> {len(sel_features)} features")

        if len(sel_features) == 0:
            print("  [WARN] No features survived AE selection – skipping fold.")
            continue

        # ── STEP 2 – RFE ranking (train only) ─────────────────────────────
        rfe_order       = step2_rfe_order(X_train_sel, y_train)
        ranked_features = [sel_features[i] for i in rfe_order]
        X_train_ranked  = X_train_sel[:, rfe_order]
        X_test_ranked   = X_test_sel[:,  rfe_order]
        print(f"  Step 2 | RFE ranking    : top-5 = {ranked_features[:5]}")

        # ── STEP 3 – IFS (train only, inner CV) ───────────────────────────
        best_k, ifs_features = step3_ifs(
            X_train_ranked, y_train, ranked_features)

        X_train_ifs = X_train_ranked[:, :best_k]
        X_test_ifs  = X_test_ranked[:,  :best_k]
        print(f"  Step 3 | IFS optimal    : k={best_k}")

        # ── STEP 4 – SBS (train only, inner CV) ───────────────────────────
        kept_cols, final_features = step4_sbs(
            X_train_ifs, y_train, ifs_features)

        X_train_final = X_train_ifs[:, kept_cols]
        X_test_final  = X_test_ifs[:,  kept_cols]
        print(f"  Step 4 | SBS refined    : n={len(final_features)}  "
              f"-> {final_features}")

        # ── STEP 5 – Fit XGBoost on TRAIN, evaluate on TEST ────────────────
        xgb_final = make_xgb(y_train)
        xgb_final.fit(X_train_final, y_train)
        y_pred = xgb_final.predict(X_test_final)

        m = {
            "fold":            fold_idx,
            "n_features":      len(final_features),
            "accuracy":        accuracy_score(y_test, y_pred),
            "f1_macro":        f1_score(y_test, y_pred,
                                        average="macro", zero_division=0),
            "precision_macro": precision_score(y_test, y_pred,
                                               average="macro",
                                               zero_division=0),
            "recall_macro":    recall_score(y_test, y_pred,
                                            average="macro", zero_division=0),
        }
        fold_metrics.append(m)
        fold_features.append(final_features)

        print(f"\n  Step 5 | Fold {fold_idx} test metrics:")
        for key in ["accuracy", "f1_macro", "precision_macro", "recall_macro"]:
            print(f"           {key:<22}: {m[key]:.4f}")

    # ── Aggregate ──────────────────────────────────────────────────────────
    if not fold_metrics:
        print("\n[ERROR] No folds produced results.")
        sys.exit(1)

    df_folds = pd.DataFrame(fold_metrics)

    print("\n" + "="*65)
    print("FINAL SUMMARY  (mean +/- std across 5 folds)")
    print("="*65)
    print(f"  {'Metric':<24} {'Mean':>8}  {'Std':>8}")
    print("  " + "-"*44)

    summary_rows = []
    for col in ["accuracy", "f1_macro", "precision_macro", "recall_macro"]:
        m = df_folds[col].mean()
        s = df_folds[col].std()
        print(f"  {col:<24} {m:>8.4f}  {s:>8.4f}")
        summary_rows.append({"Metric": col,
                              "Mean":   round(m, 4),
                              "Std":    round(s, 4)})

    print(f"\n  Mean features selected : {df_folds['n_features'].mean():.1f}")

    # ── Save ───────────────────────────────────────────────────────────────
    pd.DataFrame(summary_rows).to_csv("pipeline_metrics.csv", index=False)
    df_folds.to_csv("pipeline_fold_details.csv", index=False)
    pd.DataFrame([
        {"fold": i + 1, "feature": f}
        for i, feats in enumerate(fold_features)
        for f in feats
    ]).to_csv("fold_features.csv", index=False)

    print("\n" + "="*65)
    print("Output files:")
    print("  pipeline_metrics.csv      – mean +/- std summary")
    print("  pipeline_fold_details.csv – per-fold metrics & feature counts")
    print("  fold_features.csv         – features selected per fold")
    print("="*65)


if __name__ == "__main__":
    main()

Loading omics data ...
  mrna           : 671 samples, 11343 features
  mirna          : 671 samples, 310 features
  methylation    : 671 samples, 11189 features
  cnv            : 671 samples, 11203 features

Merged matrix : 671 samples x 34045 features

OUTER 5-FOLD CROSS-VALIDATION
Train/test split is performed FIRST; all feature selection
steps (1-4) run exclusively on the training fold.

─────────────────────────────────────────────────────────────────
  FOLD 1/5  (train=536, test=135)
─────────────────────────────────────────────────────────────────
  Step 1 | LR selection   : 34045 -> 927 features
  Step 2 | RFE ranking    : top-5 = ['mrna__GATA3', 'mrna__FOXA1', 'mrna__CEP55', 'mrna__KRT14', 'mrna__C5AR2']
  Step 3 | IFS optimal    : k=76
  Step 4 | SBS refined    : n=48  -> ['mrna__GATA3', 'mrna__CEP55', 'mrna__KRT14', 'mrna__C5AR2', 'mrna__SPAG5', 'mrna__GPR160', 'mrna__SFRP1', 'mrna__TPSAB1', 'mrna__RNF39', 'methylation__PDXK', 'mrna__ARHGAP11A', 'mrna__IGSF22', 'mrna__GPR17

## ELMO Full Pipeline Entry
Use this entrypoint when you want to run ELMO as the complete pipeline:
L1 screening -> RFE ranking -> IFS -> SBS -> XGBoost.